In [1]:
from typing import List, TypedDict
import os
import time
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_groq import ChatGroq

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END

In [2]:
load_dotenv()

# API Keys
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")


In [4]:
docs = (
    PyPDFLoader("./documents/Biology_evolution_17Pages.pdf").load() +
    PyPDFLoader("./documents/Ecosystem_11pages.pdf").load()
 
)

print(len(docs))


28


In [5]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150
).split_documents(docs)

In [6]:
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

print(len(chunks))

77


In [7]:
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=HF_TOKEN
)

c:\Development\Adv-RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
vector_store = FAISS.from_documents(chunks, embeddings)

retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':4}
)

In [9]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [10]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str


In [11]:
def retrieve(state):
    q = state["question"]
    return {"docs": retriever.invoke(q)}


In [12]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}")
    ]
)


def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({
        "question": state["question"],
        "context": context
    })
    return {"answer": out.content}

In [13]:
g = StateGraph(State)

g.add_node("retrieve", retrieve)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)

app = g.compile()

print(app)

In [16]:
res = app.invoke({
"question": "What is an temperature of moon?",
"docs": [],
"answer": ""
})

In [17]:
print(res["answer"])

I don't know. The given context is about ecosystems, energy flow, and the origin of the universe, but it does not mention the temperature of the moon.
